In [1]:
!pip install -q git+https://github.com/PyThaiNLP/pythainlp

In [2]:
# install pythainlp and ssg(subword tokenizer)
!pip install -q ssg

In [ ]:
from typing import List, Union
from pythainlp.tokenize import subword_tokenize,word_tokenize
from pythainlp.util import sound_syllable
from pythainlp.util import remove_tonemark
from pythainlp.khavee import KhaveeVerifier
import pythainlp as pythai
from pythainlp.tokenize import word_tokenize
from pythainlp.tokenize import subword_tokenize
from pythainlp.util import sound_syllable
from pythainlp.util import isthai
from pythainlp.transliterate import pronunciate
from pythainlp.spell import correct
from tqdm import tqdm
import numpy as np
import pandas as pd
kv = KhaveeVerifier()

### Word and Subword Tokenizing

In [ ]:
# split text from \n to list and drop soi word ->  splitted wak list (no soi)
def split_klong(klong_text):
  splitted_klong = []
  klong_list = klong_text.split('\n')
  klong_list = [klong for klong in klong_list if klong.strip()]
  for i in range(len(klong_list)):
    if i == 1 or i == 3 or i == 5:
      klong = klong_list[i]
      if klong[0] == ' ':
        klong = klong[1:]
      klong = klong.split(' ')
      print(f"klong: {klong}")
      splitted_klong.append(klong[0])
    else:
      splitted_klong.append(klong_list[i].replace(' ', ''))
  return splitted_klong

In [28]:
# subword tokenize wak with ssg and dict
def subword_token(wak, engine='ssg'):
  subword_tokenized = subword_tokenize(wak, engine='ssg')
  if len(subword_tokenized) != 5 and len(subword_tokenized) != 2:
      subword_tokenized = subword_tokenize(wak, engine='dict')
  return subword_tokenized

In [31]:
klong_txt = """เสียงลือเสียงเล่าอ้าง
อันใด พี่เอย
เสียงย่อมยอยศใคร
ทั่วหล้า
สองเขือพี่หลับไหล
ลืมตื่น ฤๅพี่
สองพี่คิดเองอ้า
อย่าได้ถามเผือ"""
klong_txt2 = """พระสมุทรสุดลึกล้น
คณนา
สายดิ่งทิ้งทอดมา
หยั่งได้
เขาสูงอาจวัดวา
กำหนด
จิตมนุษย์นี้ไซร้
ยากแท้หยั่งถึง
"""
print(klong_txt2+"\n\n")
splitted_klong = split_klong(klong_txt2)
print(f"splitted_klong: {splitted_klong}\n\n")
for i in range(len(splitted_klong)):
  wak = splitted_klong[i]
  subword_tokenized = subword_token(wak)
  print(f"wak: {wak} -> subword tokenized: {subword_tokenized}")

พระสมุทรสุดลึกล้น
คณนา
สายดิ่งทิ้งทอดมา
หยั่งได้
เขาสูงอาจวัดวา
กำหนด
จิตมนุษย์นี้ไซร้
ยากแท้หยั่งถึง



klong: ['คณนา']
klong: ['หยั่งได้']
klong: ['กำหนด']
splitted_klong: ['พระสมุทรสุดลึกล้น', 'คณนา', 'สายดิ่งทิ้งทอดมา', 'หยั่งได้', 'เขาสูงอาจวัดวา', 'กำหนด', 'จิตมนุษย์นี้ไซร้', 'ยากแท้หยั่งถึง']


wak: พระสมุทรสุดลึกล้น -> subword tokenized: ['พระ', 'สมุทร', 'สุด', 'ลึก', 'ล้น']
wak: คณนา -> subword tokenized: ['คณ', 'นา']
wak: สายดิ่งทิ้งทอดมา -> subword tokenized: ['สาย', 'ดิ่ง', 'ทิ้ง', 'ทอด', 'มา']
wak: หยั่งได้ -> subword tokenized: ['หยั่ง', 'ได้']
wak: เขาสูงอาจวัดวา -> subword tokenized: ['เขา', 'สูง', 'อาจ', 'วัด', 'วา']
wak: กำหนด -> subword tokenized: ['กำ', 'หนด']
wak: จิตมนุษย์นี้ไซร้ -> subword tokenized: ['จิต', 'มนุษย์', 'นี้', 'ไซ', 'ร้']
wak: ยากแท้หยั่งถึง -> subword tokenized: ['ยาก', 'แท้', 'หยั่ง', 'ถึง']


### Check Functions

#### Number of syllables check

In [ ]:

# check number of syllables -> [True, True, True, True, True, True, True, True] (len=8)
def subword_num(splitted_klong):
  checked = []
  two = [1,3,5]
  five = [0,2,4,6]
  for num in range(len(splitted_klong)):
    if num in two:
      checked.append(len(subword_token(splitted_klong[num])) == 2)
    elif num in five:
      checked.append(len(subword_token(splitted_klong[num])) == 5)
    elif num == 7:
      checked.append(len(subword_token(splitted_klong[num])) == 4)
  return checked

#### eak tou check


In [ ]:
# check what word tone is
def find_tone(word):
  char_list = [*word]
  if "่" in char_list or sound_syllable(word) == 'dead':
    return "eak or dead"
  elif "้" in char_list:
    return "tou"
  else:
    return False

In [ ]:
# check eaktou -> list[True, True, True, True, True, True, True, True] (len=8)
def check_eaktou(splitted_klong):
  checked = []
  for num in range(len(splitted_klong)):
    tokenzied_wak = subword_token(splitted_klong[num])
    if num == 0:
      checked.append(find_tone(tokenzied_wak[3]) == "eak or dead" and find_tone(tokenzied_wak[4]) == 'tou')
    elif num == 1:
      checked.append(True)
    elif num == 2:
      checked.append(find_tone(tokenzied_wak[1]) == "eak or dead")
    elif num == 3:
      checked.append(find_tone(tokenzied_wak[0]) == 'eak or dead' and find_tone(tokenzied_wak[1]) == 'tou')
    elif num == 4:
      checked.append(find_tone(tokenzied_wak[2]) == 'eak or dead')
    elif num == 5:
      checked.append(find_tone(tokenzied_wak[1]) == 'eak or dead')
    elif num == 6:
      checked.append(find_tone(tokenzied_wak[1]) == "eak or dead" and find_tone(tokenzied_wak[4]) == 'tou')
    elif num == 7:
      checked.append(find_tone(tokenzied_wak[0]) == "eak or dead" and find_tone(tokenzied_wak[1]) == 'tou')
  return checked

#### sampas check

In [ ]:
# last sound of wak from pronunciate tokenized last word of each wak
# ex [เสียงลือเสียงเล่าอ้าง] -> [อ้าง]
def sound_words(splitted_klong):
  sound_list = []
  for wak in splitted_klong:
    list_char = [*wak]
    if " " in list_char:
      wak = wak.split(" ")
      wak = wak[0]
    wak = word_tokenize(wak, engine="newmm")
    pronounce_word = pronunciate(wak[-1], engine="w2p")
    sound_list.append(pronounce_word.replace('ฺ', '').split('-')[-1])
  return sound_list

In [ ]:
# check sampas -> [True, True, True]
# [0] = sampas wak 2-3, [1] = sampas wak 2-4, [2] sampas wak 4-7
def check_sampas(sound_list):
  checked = []
  if len(sound_list) > 2:
    checked.append(kv.check_sumpus(sound_list[1],sound_list[2]))
    if len(sound_list) > 4:
      checked.append(kv.check_sumpus(sound_list[1],sound_list[4]))
      if len(sound_list) > 6:
        checked.append(kv.check_sumpus(sound_list[3],sound_list[6]))
  else:
    checked.append(True)
  return checked

#### Main Check

In [ ]:
def main_check(klong_text):
  splitted_klong = split_klong(klong_text)
  checked_subword_num = subword_num(splitted_klong)
  if False in checked_subword_num:
    false_index = checked_subword_num.index(False)
    return 'syllable format error', false_index+1
  else:
    checked_eaktou = check_eaktou(splitted_klong)
    if False in checked_eaktou:
      false_index = checked_eaktou.index(False)
      return 'eaktou format error', false_index+1
    else:
      sound_list = sound_words(splitted_klong)
      checked_sampas = check_sampas(sound_list)
      if False in checked_sampas:
        wak_sampas = ['2 and 3', '2 and 5', '4 and 7']
        return 'sampas format error', wak_sampas[checked_sampas.index(False)]
      else:
        return True

In [ ]:
def analyze_wak(wak_text):
    # 1. Tokenize into words
    words = word_tokenize(wak_text, engine="newmm")
    
    wak_data = []
    total_syllables = 0
    
    # 2. Analyze each word
    for word in words:
        # Ignore whitespace
        if word.strip() == "":
            continue
            
        spoken = pronunciate(word, engine="w2p")
        syllables = spoken.split("-")
        count = len(syllables)
        
        wak_data.append({
            "original_word": word,
            "spoken_form": spoken,
            "syllable_count": count
        })
        
        total_syllables += count

    # 3. Classify the Wak based on your rules
    classification = ""
    target_rhythm = []
    needs_manual_review = False
    
    if total_syllables == 9:
        classification = "9 Syllables"
        target_rhythm = [3, 3, 3]
    elif total_syllables == 8:
        classification = "8 Syllables"
        target_rhythm = [3, 2, 3]
    elif total_syllables == 7:
        classification = "7 Syllables"
        # Flagging for manual review as requested
        target_rhythm = [3, 2, 2] # Default assumption
        needs_manual_review = True
    else:
        classification = f"Irregular ({total_syllables} Syllables)"
        needs_manual_review = True

    return {
        "total_syllables": total_syllables,
        "classification": classification,
        "rhythm": target_rhythm,
        "needs_review": needs_manual_review,
        "word_breakdown": wak_data
    }

# --- Example Usage ---
test_wak = "ธรรมชาติสวยงามตามภูผา" # ธรรมชาติ (3) สวยงาม (2) ตาม (1) ภูผา (2) = 8
result = analyze_wak(test_wak)

print(f"Total: {result['total_syllables']} ({result['classification']})")
print(f"Rhythm: {result['rhythm']}")
print(f"Review Needed: {result['needs_review']}")
for item in result['word_breakdown']:
    print(f" - {item['original_word']} -> {item['spoken_form']} ({item['syllable_count']} beats)")